In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'
import pandas as pd

invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
print(invoices.shape)

Mounted at /content/drive
(18033, 10)


In [2]:
print(invoices['payment_status'].isnull().sum())

0


In [3]:
# Method A
print(invoices['payment_status'].value_counts())

# Method B
unpaid_count = (invoices['payment_status'] == 'Unpaid').sum()
print("Method B - Unpaid count:", unpaid_count)

payment_status
Paid              12536
Partially Paid     3660
Unpaid             1837
Name: count, dtype: int64
Method B - Unpaid count: 1837


In [4]:
total_rows = len(invoices)
unique_ids = invoices['invoice_id'].nunique()

print("Total rows:", total_rows)
print("Unique invoice_id values:", unique_ids)
print("Difference:", total_rows - unique_ids)

Total rows: 18033
Unique invoice_id values: 17836
Difference: 197


In [5]:
status_counts = invoices['payment_status'].value_counts()
print(status_counts)
print("Sum of all categories:", status_counts.sum())
print("Total rows:", len(invoices))
print("Match:", status_counts.sum() == len(invoices))

# Also check for anything unexpected hiding in the column
print("Unique values seen:", invoices['payment_status'].unique())

payment_status
Paid              12536
Partially Paid     3660
Unpaid             1837
Name: count, dtype: int64
Sum of all categories: 18033
Total rows: 18033
Match: True
Unique values seen: ['Unpaid' 'Paid' 'Partially Paid']


In [6]:
# Method A (what we've been using): boolean mask
unpaid_rate_A = (invoices['payment_status'] == 'Unpaid').sum() / len(invoices)

# Method B: groupby + normalize
unpaid_rate_B = invoices.groupby('payment_status').size()['Unpaid'] / len(invoices)

# Method C: value_counts with normalize built in
unpaid_rate_C = invoices['payment_status'].value_counts(normalize=True)['Unpaid']

print("Method A:", unpaid_rate_A)
print("Method B:", unpaid_rate_B)
print("Method C:", unpaid_rate_C)
print("All match:", unpaid_rate_A == unpaid_rate_B == unpaid_rate_C)

Method A: 0.10186879609604614
Method B: 0.10186879609604614
Method C: 0.10186879609604614
All match: True


In [7]:
unpaid = invoices[invoices['payment_status'] == 'Unpaid']

# 1. By branch
print("=== By Branch ===")
print(unpaid['branch_id'].value_counts())
print()

# 2. By customer (top 10 — looking for concentration)
print("=== Top 10 Customers by Unpaid Count ===")
print(unpaid['customer_id'].value_counts().head(10))
print()

# 3. By year (using invoice_date)
print("=== By Year ===")
print(pd.to_datetime(unpaid['invoice_date']).dt.year.value_counts().sort_index())

=== By Branch ===
branch_id
CHN001    400
KOL001    350
AHM001    303
DEL001    294
PUN001    250
HYD001    240
Name: count, dtype: int64

=== Top 10 Customers by Unpaid Count ===
customer_id
C0042    12
C0236     9
C0178     9
C0319     9
C0329     8
C0259     8
C0435     8
C0360     8
C0390     8
C0189     8
Name: count, dtype: int64

=== By Year ===
invoice_date
2019    306
2020    321
2021    293
2022    313
2023    296
2024    306
2025      2
Name: count, dtype: int64


In [8]:
invoice_dates = pd.to_datetime(invoices['invoice_date'])
print("Earliest invoice date:", invoice_dates.min())
print("Latest invoice date:", invoice_dates.max())
print("Total span (days):", (invoice_dates.max() - invoice_dates.min()).days)
print()
print("Invoices per year:")
print(invoice_dates.dt.year.value_counts().sort_index())

Earliest invoice date: 2019-01-03 00:00:00
Latest invoice date: 2025-01-13 00:00:00
Total span (days): 2202

Invoices per year:
invoice_date
2019    2975
2020    3031
2021    3026
2022    2948
2023    3035
2024    2968
2025      50
Name: count, dtype: int64


In [9]:
sample = invoices[invoices['payment_status'] == 'Unpaid'].sample(3, random_state=42)
print(sample[['invoice_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'grand_total', 'payment_status']])

       invoice_id customer_id branch_id invoice_date    due_date  grand_total  \
14894  INV-375052       C0321    HYD001   2019-11-15  2020-01-14    1236585.8   
14039  INV-594216       C0466    DEL001   2022-03-25  2022-04-24    2261062.0   
14539  INV-754139       C0288    DEL001   2024-05-08  2024-06-07    4485719.6   

      payment_status  
14894         Unpaid  
14039         Unpaid  
14539         Unpaid  


In [10]:
# Method A: boolean mask
partial_rate_A = (invoices['payment_status'] == 'Partially Paid').sum() / len(invoices)

# Method B: groupby
partial_rate_B = invoices.groupby('payment_status').size()['Partially Paid'] / len(invoices)

# Method C: value_counts normalize
partial_rate_C = invoices['payment_status'].value_counts(normalize=True)['Partially Paid']

print("Method A:", partial_rate_A)
print("Method B:", partial_rate_B)
print("Method C:", partial_rate_C)
print("All match:", partial_rate_A == partial_rate_B == partial_rate_C)

Method A: 0.2029612377308268
Method B: 0.2029612377308268
Method C: 0.2029612377308268
All match: True


In [11]:
partial = invoices[invoices['payment_status'] == 'Partially Paid']

print("=== By Branch ===")
print(partial['branch_id'].value_counts())
print()

print("=== Top 10 Customers by Partially Paid Count ===")
print(partial['customer_id'].value_counts().head(10))
print()

print("=== By Year ===")
print(pd.to_datetime(partial['invoice_date']).dt.year.value_counts().sort_index())

=== By Branch ===
branch_id
CHN001    763
KOL001    736
AHM001    611
DEL001    544
PUN001    528
HYD001    478
Name: count, dtype: int64

=== Top 10 Customers by Partially Paid Count ===
customer_id
C0475    17
C0225    15
C0462    15
C0135    14
C0242    14
C0219    14
C0045    14
C0008    14
C0074    14
C0178    14
Name: count, dtype: int64

=== By Year ===
invoice_date
2019    577
2020    638
2021    636
2022    595
2023    620
2024    584
2025     10
Name: count, dtype: int64


In [12]:
sample = invoices[invoices['payment_status'] == 'Partially Paid'].sample(3, random_state=42)
print(sample[['invoice_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'grand_total', 'payment_status']])

      invoice_id customer_id branch_id invoice_date    due_date  grand_total  \
1431  INV-829749       C0246    DEL001   2020-01-25  2020-02-24    5674611.0   
1358  INV-114729       C0478    DEL001   2019-07-04  2019-08-03    5936809.0   
735   INV-987366       C0314    KOL001   2023-04-14  2023-04-29     470390.4   

      payment_status  
1431  Partially Paid  
1358  Partially Paid  
735   Partially Paid  
